# 00 · Download compact HTEM process records and substrate grids

**Collect as much public data as the API exposes**, not just libraries with optical/electrical data.
Retain deposition recipes, per-library positions, substrate `x_mm`/`y_mm`, local `absolute_temp_c`, thickness, conductivity,
and optional direct bandgap. No spectra, measured composition, contacts, or full instrument records
are retained. Missing properties remain missing here; cohort selection belongs in notebook 01.

The working API is `https://htem-api.nlr.gov/api`. Its **scalar batch endpoint**
`/sample_library/prop?ids=...` supplies aligned position/coordinate/property arrays. These are
substrate-map coordinates, **not** the sample endpoint's `xyz_mm` stage/focus coordinates.
The sample-level temperature can vary across a library; it must not be replaced by the library's
initial temperature. Old compact scalar caches automatically refresh once to retrieve this field.
The key is **(sample_library_id, position)**; position 1 in one deposition is not another observation's ID.

Sources: [HTEM paper](https://doi.org/10.1038/sdata.2018.53),
[current HTEM site and API documentation](https://htem.nlr.gov/),
[official historical examples](https://github.com/NatLabRockies/htem-api-examples).

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE = Path.cwd()
if not (HERE / 'htem_workshop.py').exists():
    HERE = HERE / '03_htem_conductivity'
if not (HERE / 'htem_workshop.py').exists():
    raise FileNotFoundError('Open this notebook from the workshop root or HTEM module folder.')
sys.path.insert(0, str(HERE))
import htem_workshop as htem
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
SEED = 42

## Acquire all public libraries, with compact resumable checkpoints

Default: **live acquisition with no library cap or measured-property filter**. Cached compact
responses are reused. Set `REFRESH=True` to refresh them, or choose explicit `snapshot` mode for
offline use. Network/source problems are reported, not disguised as a fresh successful full crawl.

Most collection uses one recipe request per library plus batches of scalar maps. Rare libraries
unsupported by the scalar endpoint fall back to individual sample responses; spectra in those
responses are discarded **in memory before any cache is written**. If the substrate XY map is
unavailable it stays missing—we never substitute stage coordinates or guess a 4×11 grid.

In [2]:
import shutil
ACQUISITION_MODE = 'live_api'  # explicit 'snapshot' for offline use
LIBRARY_IDS = None            # None = EVERY public library; no cap
REFRESH = False               # reuse compact per-library checkpoints
WORKERS = 4                  # bounded recipe-download concurrency
ACQUISITION = HERE / 'outputs/htem_acquisition'
ACQUISITION.mkdir(parents=True, exist_ok=True)
RAW_FILE = ACQUISITION / 'htem_process_raw.parquet'
SNAPSHOT = HERE / 'data/htem_spatial_snapshot.parquet'
LIBRARIES = ACQUISITION / 'htem_libraries.parquet'

if ACQUISITION_MODE == 'live_api':
    raw = htem.download_htem(RAW_FILE, ACQUISITION / 'compact_http_cache_v2',
                             library_ids=LIBRARY_IDS, workers=WORKERS, refresh=REFRESH)
    provenance = json.loads(RAW_FILE.with_suffix('.provenance.json').read_text(encoding='utf-8'))
    # Distribute the compact result, never raw HTTP payloads or spectra.
    shutil.copy2(RAW_FILE, SNAPSHOT)
    shutil.copy2(LIBRARIES, HERE / 'data/htem_spatial_libraries.parquet')
    htem.write_json(SNAPSHOT.with_suffix('.provenance.json'), provenance)
elif ACQUISITION_MODE == 'snapshot':
    provenance = json.loads(SNAPSHOT.with_suffix('.provenance.json').read_text(encoding='utf-8'))
    assert htem.sha256(SNAPSHOT) == provenance['sha256'], 'Snapshot checksum mismatch.'
    shutil.copy2(SNAPSHOT, RAW_FILE)
    shutil.copy2(HERE / 'data/htem_spatial_libraries.parquet', LIBRARIES)
    raw = pd.read_parquet(RAW_FILE)
    provenance = {**provenance, 'mode': 'explicit_snapshot_reuse_NOT_a_new_download'}
    htem.write_json(RAW_FILE.with_suffix('.provenance.json'), provenance)
else:
    raise ValueError("Choose 'live_api' or 'snapshot'.")
assert raw.columns.tolist() == htem.RAW_FIELDS
assert not raw.duplicated(['sample_library_id', 'position']).any()
print(f"Mode: {provenance['mode']}; complete requested sample coverage: {provenance['complete']}")
print(f"Public libraries: {provenance['public_libraries']:,}; recipes: {provenance['recipe_libraries']:,}")
print(f"Retained points: {len(raw):,}; compact Parquet: {RAW_FILE.stat().st_size / 1e6:.2f} MB")
print('Rows without substrate coordinates:', provenance['rows_missing_substrate_coordinates'])
display(raw[['sample_library_id', 'position', 'x_mm', 'y_mm', 'absolute_temp_c', *htem.LABEL_FIELDS]].count().rename('available_values').to_frame())
display(pd.DataFrame(provenance['failures']))
display(pd.DataFrame(provenance['sample_count_gaps']))

Recipes checked 1/1891


Recipes checked 100/1891


Recipes checked 200/1891


Recipes checked 300/1891


Recipes checked 400/1891


Recipes checked 500/1891


... (91 outputs left out)

Mode: live_scalar_api; complete requested sample coverage: False
Public libraries: 1,891; recipes: 1,891
Retained points: 82,644; compact Parquet: 0.96 MB
Rows without substrate coordinates: 44


,available_values
sample_library_id,82644
position,82644
x_mm,82600
y_mm,82600
absolute_temp_c,66264
thickness,54543
opt_direct_bandgap,24797
fpm_conductivity,13676


,library_id,stage,recovered_points,unavailable_samples
0,10941,individual_scalar_fallback,0,"[{'sample_id': 370941, 'error': '400 Client Er..."
1,12922,individual_scalar_fallback,0,"[{'sample_id': 399408, 'error': '400 Client Er..."
2,12930,individual_scalar_fallback,0,"[{'sample_id': 396447, 'error': '400 Client Er..."


,library_id,expected_samples,scalar_positions
0,10941,44,0
1,12922,44,0
2,12930,44,0


## Inspect identity and geometry before cleansing

All process recipes—including ones with no usable targets—are retained in `htem_libraries.parquet`.
The point table retains scalar labels and coordinates even when other labels are missing.
`id` is a human-readable **library:position composite key**, not the API's numeric sample ID.
A geometry cannot be recovered by sorting those API IDs or by assuming all grids have 44 points.

In [3]:
display(raw[['id', 'sample_library_id', 'position', 'x_mm', 'y_mm',
             'deposition_compounds', 'deposition_power', 'thickness', 'fpm_conductivity']].head(8))
display(raw.groupby('sample_library_id').size().value_counts().rename_axis('points_per_library').rename('libraries').to_frame())
print('Only allowlisted process fields, identities, substrate XY, and scalar property labels are saved.')
print('Missing XY is NOT filled from xyz_mm. Missing targets are NOT imputed during acquisition.')

,id,sample_library_id,position,x_mm,y_mm,deposition_compounds,deposition_power,thickness,fpm_conductivity
0,2952:1,2952,1,5.4,6.65,"[None, None, None]","[None, None, None]",0.1123,NaN
1,2952:2,2952,2,9.4,6.65,"[None, None, None]","[None, None, None]",0.1192,NaN
2,2952:3,2952,3,13.4,6.65,"[None, None, None]","[None, None, None]",0.1259,NaN
3,2952:4,2952,4,17.4,6.65,"[None, None, None]","[None, None, None]",0.1295,NaN
4,2952:5,2952,5,21.4,6.65,"[None, None, None]","[None, None, None]",0.1331,NaN
5,2952:6,2952,6,25.4,6.65,"[None, None, None]","[None, None, None]",0.1339,NaN
6,2952:7,2952,7,29.4,6.65,"[None, None, None]","[None, None, None]",0.1323,NaN
7,2952:8,2952,8,33.4,6.65,"[None, None, None]","[None, None, None]",0.1291,NaN


,libraries
points_per_library,
44,1873
11,5
177,1


Only allowlisted process fields, identities, substrate XY, and scalar property labels are saved.
Missing XY is NOT filled from xyz_mm. Missing targets are NOT imputed during acquisition.
